# 01 — Building Evaluation Datasets

**Track:** Intermediate · **Stage:** Evaluation

Before you can evaluate a RAG system, you need a **Golden Dataset**. A golden dataset consists of pairs of `(question, ground_truth_answer, ground_truth_context)`.

Relying on human experts to write 500 questions is slow and expensive. In this notebook, we will use an LLM to **synthetically generate** an evaluation dataset directly from our documents, utilizing techniques popularized by frameworks like **Ragas**.

## Setup: LangChain

We will use LangChain to parse our documents and a mock LLM to simulate the generation of test questions.

In [ ]:
# !pip install langchain langchain-core

import json
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_community.llms.fake import FakeListLLM

## 1. The Source Corpus

We have a few policy documents. We want to generate questions that can only be answered if these documents are retrieved.

In [ ]:
corpus = [
    Document(
        page_content="PolicyAssist Standard: Water damage coverage is capped at $50,000 for residential properties unless the flood rider (FR-99) is purchased.",
        metadata={"source": "residential_policy.md", "id": "doc_01"}
    ),
    Document(
        page_content="PolicyAssist Auto: Comprehensive auto insurance does not cover mechanical failure, such as a blown engine or transmission failure.",
        metadata={"source": "auto_policy.md", "id": "doc_02"}
    )
]

## 2. Generating Simple Questions

We prompt an LLM to read a single chunk of text and generate a question that can be answered *exclusively* using that text.

In [ ]:
generation_template = """
You are an expert underwriter. Read the following policy text.
Generate 1 question that can be answered using this text, and provide the exact answer.
Output strictly in JSON format: {{"question": "...", "answer": "..."}}

Policy Text: {context}
"""
generation_prompt = ChatPromptTemplate.from_template(generation_template)

# Simulating the LLM generating JSON
mock_responses = [
    json.dumps({"question": "What is the cap for residential water damage coverage?", "answer": "$50,000 unless the FR-99 rider is purchased."}),
    json.dumps({"question": "Does comprehensive auto cover a blown engine?", "answer": "No, mechanical failures are not covered."}) 
]
generation_llm = FakeListLLM(responses=mock_responses)
generation_chain = generation_prompt | generation_llm | JsonOutputParser()

print("--- Synthetically Generated Golden Dataset ---")
golden_dataset = []
for doc in corpus:
    result = generation_chain.invoke({"context": doc.page_content})
    result["ground_truth_context"] = doc.page_content
    result["ground_truth_doc_id"] = doc.metadata["id"]
    golden_dataset.append(result)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}\n")

## 3. Advanced Generation: Multi-hop and Adversarial

A good evaluation dataset cannot only have easy questions. Frameworks like **Ragas** automatically mutate questions to make them harder:
- **Reasoning/Multi-hop:** Combine two chunks and ask a question requiring both.
- **Conditioning:** Add a condition (e.g., "If the customer lives in Florida, what is the cap?").
- **Adversarial:** Ask a question that sounds similar to the text but is actually unanswerable (testing abstention).

In [ ]:
mutation_template = """
Take the following simple question and make it harder by adding a hypothetical customer scenario.
Output only the mutated question.
Original Question: {question}
"""
mutation_prompt = ChatPromptTemplate.from_template(mutation_template)
mutation_llm = FakeListLLM(responses=[
    "My basement flooded and caused $80,000 in damage, but I don't have the FR-99 rider. How much will be covered?"
])
mutation_chain = mutation_prompt | mutation_llm | StrOutputParser()

print("--- Mutated Hard Question ---")
hard_question = mutation_chain.invoke({"question": golden_dataset[0]["question"]})
print(f"Hard Q: {hard_question}")
print(f"Expected A: {golden_dataset[0]['answer']}")

## Reflection

1. **Quality Control:** Generating synthetic datasets is fast, but LLMs can write bad questions. Always have a human review a random 10% sample of the generated dataset before relying on it for CI/CD gates.
2. **Frameworks:** Instead of writing these prompts manually, you can use `ragas.testset.generator.TestsetGenerator` to automatically generate reasoning, conditional, and conversational questions across a large corpus.